First_project_day1_Bindu: Dealing with the missing values

In [2]:
import pandas as pd

In [3]:
url = r"../data/raw/imdb_top_1000.csv"
df_clean = pd.read_csv(url) #df is named as df_clean to be convenient while merging with other notebook
df_clean.head()


,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,"28,341,469"
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,"57,300,000"
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,"4,360,000"


In [4]:
df_clean.columns = df_clean.columns.str.strip().str.lower().str.replace(" ", "_")

In [5]:
df_clean = df_clean.rename(columns={"series_title": "movie_title",
                        "no_of_votes": "number_of_votes",
                        "gross": "gross_revenue"})

In [6]:
#Missing values
df_clean.isna().sum()

poster_link          0
movie_title          0
released_year        0
certificate        101
runtime              0
genre                0
imdb_rating          0
overview             0
meta_score         157
director             0
star1                0
star2                0
star3                0
star4                0
number_of_votes      0
gross_revenue      169
dtype: int64

In [7]:
#to know data types of each column
df_clean.dtypes

poster_link         object
movie_title         object
released_year       object
certificate         object
runtime             object
genre               object
imdb_rating        float64
overview            object
meta_score         float64
director            object
star1               object
star2               object
star3               object
star4               object
number_of_votes      int64
gross_revenue       object
dtype: object

In [8]:
# gives the different certificate with count, including NaN
df_clean["certificate"].value_counts(dropna=False)  

certificate
U           234
A           197
UA          175
R           146
NaN         101
PG-13        43
PG           37
Passed       34
G            12
Approved     11
TV-PG         3
GP            2
TV-14         1
16            1
TV-MA         1
Unrated       1
U/A           1
Name: count, dtype: int64

In [9]:
# Replace NaN values with Unrated
df_clean["certificate"]= df_clean["certificate"].fillna("Unrated")

In [11]:
# ratings from India, US and TV are grouped into 4 audience levels. 

age_groups = {"U": "All ages",
              "G": "All ages",
              "Passed": "All ages",
              "Approved": "All ages",
              "UA": "Family/Teen",
              "U/A": "Family/Teen",
              "PG-13": "Family/Teen",
              "PG": "Family/Teen",
              "GP": "Family/Teen",
              "TV-PG": "Family/Teen",
              "TV-14": "Family/Teen",
              "A": "Adult",
              "R": "Adult",
              "TV-MA": "Adult",
              "16": "Adult",
              "Unrated": "Unrated"    
    }
df_clean["audience"] = df_clean["certificate"].map(age_groups)


0            Adult
1            Adult
2      Family/Teen
3            Adult
4         All ages
          ...     
995          Adult
996       All ages
997       All ages
998        Unrated
999        Unrated
Name: audience, Length: 1000, dtype: object

In [31]:
df_clean["audience"].value_counts() # ratings from India, US and TV are grouped into 4 audience levels. 

audience
Adult          345
All ages       291
Family/Teen    262
Unrated        102
Name: count, dtype: int64

In [32]:
df_clean["audience"].isna().sum() # no NaN values

np.int64(0)

In [33]:
df_clean["meta_score"].unique() 

array([ 80., 100.,  84.,  90.,  96.,  94.,  74.,  66.,  92.,  82.,  87.,
        73.,  83.,  nan,  79.,  91.,  61.,  59.,  65.,  85.,  98.,  89.,
        88.,  57.,  67.,  62.,  77.,  64.,  75.,  97.,  99.,  78.,  68.,
        81.,  95.,  76.,  69.,  55.,  70.,  58.,  86.,  71.,  63.,  93.,
        72.,  60.,  47.,  49.,  50.,  33.,  54.,  56.,  51.,  53.,  48.,
        44.,  45.,  40.,  52.,  28.,  36.,  46.,  30.,  41.])

In [34]:
#  which films had no critic score 
df_clean["meta_missing"] = df_clean["meta_score"].isna()

# Fill missing scores with the median score of films with the same IMDb rating
df_clean["meta_score"] = df_clean["meta_score"].fillna(
    df_clean.groupby("imdb_rating")["meta_score"].transform("median"))


In [ ]:
# real = df_clean[~df_clean["meta_missing"]] 

In [35]:
df_clean["meta_score"].isna().sum()

np.int64(0)

In [36]:
df_clean["gross_revenue"].isna().sum() ##169 films have no US box office data and were left out of revenue analysis. For popularity among all 1000 films, use No_of_Votes.

np.int64(169)

In [37]:
df_clean["gross_revenue"] = (df_clean["gross_revenue"]
                     .str.strip()
                     .str.replace(",", "", regex=False))
df_clean["gross_revenue"] = pd.to_numeric(df_clean["gross_revenue"], errors="coerce")

#  millions, easier to read in charts
df_clean["gross_revenue"] = df_clean["gross_revenue"] / 1_000_000


In [38]:
df_clean["gross_revenue"].dtype        

dtype('float64')

In [39]:
df_clean["gross_revenue"].isna().sum()  

np.int64(169)

In [40]:
df_clean 

,poster_link,movie_title,released_year,certificate,runtime,genre,imdb_rating,overview,meta_score,director,star1,star2,star3,star4,number_of_votes,gross_revenue,audience,meta_missing
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,28.341469,Adult,False
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,134.966411,Adult,False
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,534.858444,Family/Teen,False
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,57.300000,Adult,False
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,4.360000,All ages,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,https://m.media-amazon.com/images/M/MV5BNGEwMT...,Breakfast at Tiffany's,1961,A,115 min,"Comedy, Drama, Romance",7.6,A young New York socialite becomes interested ...,76.0,Blake Edwards,Audrey Hepburn,George Peppard,Patricia Neal,Buddy Ebsen,166544,NaN,Adult,False
996,https://m.media-amazon.com/images/M/MV5BODk3Yj...,Giant,1956,G,201 min,"Drama, Western",7.6,Sprawling epic covering the life of a Texas ca...,84.0,George Stevens,Elizabeth Taylor,Rock Hudson,James Dean,Carroll Baker,34075,NaN,All ages,False
997,https://m.media-amazon.com/images/M/MV5BM2U3Yz...,From Here to Eternity,1953,Passed,118 min,"Drama, Romance, War",7.6,"In Hawaii in 1941, a private is cruelly punish...",85.0,Fred Zinnemann,Burt Lancaster,Montgomery Clift,Deborah Kerr,Donna Reed,43374,30.500000,All ages,False
998,https://m.media-amazon.com/images/M/MV5BZTBmMj...,Lifeboat,1944,Unrated,97 min,"Drama, War",7.6,Several survivors of a torpedoed merchant ship...,78.0,Alfred Hitchcock,Tallulah Bankhead,John Hodiak,Walter Slezak,William Bendix,26471,NaN,Unrated,False
